In [1]:
import cv2
import torch
import torch.nn as nn
import numpy as np

# Define MC-CNN model (copy your exact model code here)
class MC_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.column1 = nn.Sequential(
            nn.Conv2d(3, 8, 9, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(32, 16, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(16, 8, 7, padding='same'),
            nn.ReLU(),
        )
        self.column2 = nn.Sequential(
            nn.Conv2d(3, 10, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(10, 20, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(20, 40, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(40, 20, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(20, 10, 5, padding='same'),
            nn.ReLU(),
        )
        self.column3 = nn.Sequential(
            nn.Conv2d(3, 12, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(48, 24, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(24, 12, 3, padding='same'),
            nn.ReLU(),
        )
        self.fusion_layer = nn.Sequential(
            nn.Conv2d(30, 1, 1, padding=0),
        )
    def forward(self, x):
        x1 = self.column1(x)
        x2 = self.column2(x)
        x3 = self.column3(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.fusion_layer(x)
        return x

def preprocess_frame(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (1024, 768))  # Match your model training input size
    img = resized.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # HWC to CHW
    tensor = torch.from_numpy(img).unsqueeze(0)  # Add batch dimension
    return tensor

def main():
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    model = MC_CNN().to(device)
    model.load_state_dict(torch.load("crowd_counting.pth", map_location=device), strict=False)
    model.eval()
    print("Model loaded.")

    input_video_path = r"C:\Users\lekha\Videos\Screen Recordings\Screen Recording 2025-12-02 193345.mp4"
    output_video_path = "output_with_counts.mp4"

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error opening video file.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        input_tensor = preprocess_frame(frame).to(device)
        with torch.no_grad():
            density_map = model(input_tensor)
        density_map = density_map.squeeze().cpu().numpy()
        count = float(density_map.sum())

        # Display count on frame
        cv2.putText(frame, f"Count: {count:.2f}", (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

        out.write(frame)

    cap.release()
    out.release()
    print(f"Processing complete, saved output video at {output_video_path}")

if __name__ == "__main__":
    main()


Using device: cpu
Model loaded.
Processing complete, saved output video at output_with_counts.mp4


In [2]:
import cv2
import torch
import torch.nn as nn
import numpy as np

# ====== CONFIG ======
CELL_THRESHOLD = 50.0  # change this to whatever per-cell limit you want

# ====== MC-CNN MODEL ======
class MC_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.column1 = nn.Sequential(
            nn.Conv2d(3, 8, 9, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(32, 16, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(16, 8, 7, padding='same'),
            nn.ReLU(),
        )
        self.column2 = nn.Sequential(
            nn.Conv2d(3, 10, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(10, 20, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(20, 40, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(40, 20, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(20, 10, 5, padding='same'),
            nn.ReLU(),
        )
        self.column3 = nn.Sequential(
            nn.Conv2d(3, 12, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(48, 24, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(24, 12, 3, padding='same'),
            nn.ReLU(),
        )
        self.fusion_layer = nn.Sequential(
            nn.Conv2d(30, 1, 1, padding=0),
        )

    def forward(self, x):
        x1 = self.column1(x)
        x2 = self.column2(x)
        x3 = self.column3(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.fusion_layer(x)
        return x

def preprocess_patch(patch_bgr, target_size=(1024, 768)):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, target_size)  # (W, H)
    img = resized.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # HWC -> CHW
    tensor = torch.from_numpy(img).unsqueeze(0)  # [1,3,H,W]
    return tensor

def density_to_heatmap(dmap_np, out_w, out_h):
    if dmap_np.max() > 0:
        norm = dmap_np / (dmap_np.max() + 1e-6)
    else:
        norm = dmap_np
    heat = (norm * 255).astype(np.uint8)
    heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat_color = cv2.resize(heat_color, (out_w, out_h))
    return heat_color

def main():
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    model = MC_CNN().to(device)
    model.load_state_dict(torch.load("crowd_counting.pth", map_location=device), strict=False)
    model.eval()
    print("Model loaded.")

    input_video_path = r"C:\Users\lekha\Videos\Screen Recordings\Screen Recording 2025-12-02 193345.mp4"
    output_video_path = "output_with_counts_4cells_threshold.mp4"

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error opening video file.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (frame_width, frame_height)
    )

    mid_x = frame_width // 2
    mid_y = frame_height // 2

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Split into 4 cells
        tl = frame[0:mid_y, 0:mid_x]
        tr = frame[0:mid_y, mid_x:frame_width]
        bl = frame[mid_y:frame_height, 0:mid_x]
        br = frame[mid_y:frame_height, mid_x:frame_width]

        cells = [tl, tr, bl, br]
        cell_counts = []
        cell_heats = []

        for patch in cells:
            input_tensor = preprocess_patch(patch).to(device)
            with torch.no_grad():
                dmap = model(input_tensor)
            dmap_np = dmap.squeeze().cpu().numpy()
            count = float(dmap_np.sum())
            cell_counts.append(count)

            h, w, _ = patch.shape
            heat = density_to_heatmap(dmap_np, w, h)
            cell_heats.append(heat)

        overlay = frame.copy()

        # Overlay heatmaps
        overlay[0:mid_y, 0:mid_x] = cv2.addWeighted(
            frame[0:mid_y, 0:mid_x], 0.6, cell_heats[0], 0.4, 0
        )
        overlay[0:mid_y, mid_x:frame_width] = cv2.addWeighted(
            frame[0:mid_y, mid_x:frame_width], 0.6, cell_heats[1], 0.4, 0
        )
        overlay[mid_y:frame_height, 0:mid_x] = cv2.addWeighted(
            frame[mid_y:frame_height, 0:mid_x], 0.6, cell_heats[2], 0.4, 0
        )
        overlay[mid_y:frame_height, mid_x:frame_width] = cv2.addWeighted(
            frame[mid_y:frame_height, mid_x:frame_width], 0.6, cell_heats[3], 0.4, 0
        )

        # Grid lines
        cv2.line(overlay, (mid_x, 0), (mid_x, frame_height), (0, 0, 255), 2)
        cv2.line(overlay, (0, mid_y), (frame_width, mid_y), (0, 0, 255), 2)

        # Red text
        color = (0, 0, 255)

        # Per-cell counts
        cv2.putText(overlay, f"C1: {cell_counts[0]:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C2: {cell_counts[1]:.1f}", (mid_x + 20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C3: {cell_counts[2]:.1f}", (20, mid_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C4: {cell_counts[3]:.1f}", (mid_x + 20, mid_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)

        total_count = sum(cell_counts)
        cv2.putText(overlay, f"Total: {total_count:.1f}", (20, frame_height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)

        # ----- Overcrowding check -----
        overcrowded_cells = [i+1 for i, c in enumerate(cell_counts) if c > CELL_THRESHOLD]
        if overcrowded_cells:
            warning_text = f"WARNING: Overcrowding in cells: {overcrowded_cells}"
            cv2.putText(overlay, warning_text, (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)

        out.write(overlay)

    cap.release()
    out.release()
    print(f"Processing complete, saved output video at {output_video_path}")

if __name__ == "__main__":
    main()


Using device: cpu
Model loaded.
Processing complete, saved output video at output_with_counts_4cells_threshold.mp4


In [3]:
#saved video
import cv2
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO  # NEW: YOLOv8 for person detection

# ====== CONFIG ======
CELL_THRESHOLD = 50.0  # per-cell density limit
YOLO_MODEL_PATH = "yolov8n.pt"  # uses COCO pretrained; person class id = 0

# ====== MC-CNN MODEL ======
class MC_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.column1 = nn.Sequential(
            nn.Conv2d(3, 8, 9, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(32, 16, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(16, 8, 7, padding='same'),
            nn.ReLU(),
        )
        self.column2 = nn.Sequential(
            nn.Conv2d(3, 10, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(10, 20, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(20, 40, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(40, 20, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(20, 10, 5, padding='same'),
            nn.ReLU(),
        )
        self.column3 = nn.Sequential(
            nn.Conv2d(3, 12, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(48, 24, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(24, 12, 3, padding='same'),
            nn.ReLU(),
        )
        self.fusion_layer = nn.Sequential(
            nn.Conv2d(30, 1, 1, padding=0),
        )

    def forward(self, x):
        x1 = self.column1(x)
        x2 = self.column2(x)
        x3 = self.column3(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.fusion_layer(x)
        return x

def preprocess_patch(patch_bgr, target_size=(1024, 768)):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, target_size)  # (W, H)
    img = resized.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # HWC -> CHW
    tensor = torch.from_numpy(img).unsqueeze(0)  # [1,3,H,W]
    return tensor

def density_to_heatmap(dmap_np, out_w, out_h):
    if dmap_np.max() > 0:
        norm = dmap_np / (dmap_np.max() + 1e-6)
    else:
        norm = dmap_np
    heat = (norm * 255).astype(np.uint8)
    heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat_color = cv2.resize(heat_color, (out_w, out_h))
    return heat_color

def main():
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Load MC-CNN
    model = MC_CNN().to(device)
    model.load_state_dict(torch.load("crowd_counting.pth", map_location=device), strict=False)
    model.eval()
    print("MC-CNN model loaded.")

    # Load YOLO person detector (CPU or GPU automatically)
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("YOLO model loaded for person boxes.")

    input_video_path = r"C:\Users\lekha\Videos\Screen Recordings\Screen Recording 2025-12-02 193345.mp4"
    output_video_path = "output_with_counts_4cells_threshold_boxes.mp4"

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error opening video file.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (frame_width, frame_height)
    )

    mid_x = frame_width // 2
    mid_y = frame_height // 2

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # --------- 1) MC-CNN: 4-cell density + heatmaps ---------
        tl = frame[0:mid_y, 0:mid_x]
        tr = frame[0:mid_y, mid_x:frame_width]
        bl = frame[mid_y:frame_height, 0:mid_x]
        br = frame[mid_y:frame_height, mid_x:frame_width]

        cells = [tl, tr, bl, br]
        cell_counts = []
        cell_heats = []

        for patch in cells:
            input_tensor = preprocess_patch(patch).to(device)
            with torch.no_grad():
                dmap = model(input_tensor)
            dmap_np = dmap.squeeze().cpu().numpy()
            count = float(dmap_np.sum())
            cell_counts.append(count)

            h, w, _ = patch.shape
            heat = density_to_heatmap(dmap_np, w, h)
            cell_heats.append(heat)

        overlay = frame.copy()

        # Overlay heatmaps
        overlay[0:mid_y, 0:mid_x] = cv2.addWeighted(
            frame[0:mid_y, 0:mid_x], 0.6, cell_heats[0], 0.4, 0
        )
        overlay[0:mid_y, mid_x:frame_width] = cv2.addWeighted(
            frame[0:mid_y, mid_x:frame_width], 0.6, cell_heats[1], 0.4, 0
        )
        overlay[mid_y:frame_height, 0:mid_x] = cv2.addWeighted(
            frame[mid_y:frame_height, 0:mid_x], 0.6, cell_heats[2], 0.4, 0
        )
        overlay[mid_y:frame_height, mid_x:frame_width] = cv2.addWeighted(
            frame[mid_y:frame_height, mid_x:frame_width], 0.6, cell_heats[3], 0.4, 0
        )

        # Grid lines
        cv2.line(overlay, (mid_x, 0), (mid_x, frame_height), (0, 0, 255), 2)
        cv2.line(overlay, (0, mid_y), (frame_width, mid_y), (0, 0, 255), 2)

        # --------- 2) YOLO: draw green boxes for each person ---------
        # Run YOLO on original frame (BGR→RGB inside YOLO)
        results = yolo_model.predict(frame, verbose=False)
        boxes = results[0].boxes

        for box in boxes:
            cls = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            # COCO class 0 = person
            if cls != 0:
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            # draw green box
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # optional: draw confidence
            cv2.putText(overlay, f"{conf:.2f}", (x1, max(y1-5, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

        # --------- 3) Draw counts and warnings (red text) ---------
        color = (0, 0, 255)

        cv2.putText(overlay, f"C1: {cell_counts[0]:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C2: {cell_counts[1]:.1f}", (mid_x + 20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C3: {cell_counts[2]:.1f}", (20, mid_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C4: {cell_counts[3]:.1f}", (mid_x + 20, mid_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)

        total_count = sum(cell_counts)
        cv2.putText(overlay, f"Total: {total_count:.1f}", (20, frame_height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)

        overcrowded_cells = [i+1 for i, c in enumerate(cell_counts) if c > CELL_THRESHOLD]
        if overcrowded_cells:
            warning_text = f"WARNING: Overcrowding in cells: {overcrowded_cells}"
            cv2.putText(overlay, warning_text, (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)

        out.write(overlay)

    cap.release()
    out.release()
    print(f"Processing complete, saved output video at {output_video_path}")

if __name__ == "__main__":
    main()


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\lekha\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cpu
MC-CNN model loaded.
YOLO model loaded for person boxes.
Processing complete, saved output video at output_with_counts_4cells_threshold_boxes.mp4


In [ ]:
#real-time
import cv2
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO  # YOLOv8 for person detection

# ====== CONFIG ======
CELL_THRESHOLD = 50.0              # per-cell density limit
YOLO_MODEL_PATH = "yolov8n.pt"     # COCO pretrained; class 0 = person
YOLO_CONF_THRESH = 0.15            # lower value -> more sensitive, more false positives

# ====== MC-CNN MODEL ======
class MC_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.column1 = nn.Sequential(
            nn.Conv2d(3, 8, 9, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(32, 16, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(16, 8, 7, padding='same'),
            nn.ReLU(),
        )
        self.column2 = nn.Sequential(
            nn.Conv2d(3, 10, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(10, 20, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(20, 40, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(40, 20, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(20, 10, 5, padding='same'),
            nn.ReLU(),
        )
        self.column3 = nn.Sequential(
            nn.Conv2d(3, 12, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(48, 24, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(24, 12, 3, padding='same'),
            nn.ReLU(),
        )
        self.fusion_layer = nn.Sequential(
            nn.Conv2d(30, 1, 1, padding=0),
        )

    def forward(self, x):
        x1 = self.column1(x)
        x2 = self.column2(x)
        x3 = self.column3(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.fusion_layer(x)
        return x

def preprocess_patch(patch_bgr, target_size=(1024, 768)):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, target_size)  # (W, H)
    img = resized.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # HWC -> CHW
    tensor = torch.from_numpy(img).unsqueeze(0)  # [1,3,H,W]
    return tensor

def density_to_heatmap(dmap_np, out_w, out_h):
    if dmap_np.max() > 0:
        norm = dmap_np / (dmap_np.max() + 1e-6)
    else:
        norm = dmap_np
    heat = (norm * 255).astype(np.uint8)
    heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat_color = cv2.resize(heat_color, (out_w, out_h))
    return heat_color

def main():
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Load MC-CNN
    model = MC_CNN().to(device)
    model.load_state_dict(torch.load("crowd_counting.pth", map_location=device), strict=False)
    model.eval()
    print("MC-CNN model loaded.")

    # Load YOLO person detector
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("YOLO model loaded for person boxes.")

    input_video_path = r"C:\Users\lekha\Videos\Screen Recordings\Screen Recording 2025-12-02 193345.mp4"
    output_video_path = "output_with_counts_4cells_threshold_boxes_realtime.mp4"

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error opening video file.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps if fps > 0 else 20,
        (frame_width, frame_height)
    )

    mid_x = frame_width // 2
    mid_y = frame_height // 2

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # --------- 1) MC-CNN: 4-cell density + heatmaps ---------
        tl = frame[0:mid_y, 0:mid_x]
        tr = frame[0:mid_y, mid_x:frame_width]
        bl = frame[mid_y:frame_height, 0:mid_x]
        br = frame[mid_y:frame_height, mid_x:frame_width]

        cells = [tl, tr, bl, br]
        cell_counts = []
        cell_heats = []

        for patch in cells:
            input_tensor = preprocess_patch(patch).to(device)
            with torch.no_grad():
                dmap = model(input_tensor)
            dmap_np = dmap.squeeze().cpu().numpy()
            count = float(dmap_np.sum())
            cell_counts.append(count)

            h, w, _ = patch.shape
            heat = density_to_heatmap(dmap_np, w, h)
            cell_heats.append(heat)

        overlay = frame.copy()

        # Overlay heatmaps
        overlay[0:mid_y, 0:mid_x] = cv2.addWeighted(
            frame[0:mid_y, 0:mid_x], 0.6, cell_heats[0], 0.4, 0
        )
        overlay[0:mid_y, mid_x:frame_width] = cv2.addWeighted(
            frame[0:mid_y, mid_x:frame_width], 0.6, cell_heats[1], 0.4, 0
        )
        overlay[mid_y:frame_height, 0:mid_x] = cv2.addWeighted(
            frame[mid_y:frame_height, 0:mid_x], 0.6, cell_heats[2], 0.4, 0
        )
        overlay[mid_y:frame_height, mid_x:frame_width] = cv2.addWeighted(
            frame[mid_y:frame_height, mid_x:frame_width], 0.6, cell_heats[3], 0.4, 0
        )

        # Grid lines
        cv2.line(overlay, (mid_x, 0), (mid_x, frame_height), (0, 0, 255), 2)
        cv2.line(overlay, (0, mid_y), (frame_width, mid_y), (0, 0, 255), 2)

        # --------- 2) YOLO: draw green boxes for each person ---------
        results = yolo_model.predict(frame, conf=YOLO_CONF_THRESH, verbose=False)
        boxes = results[0].boxes

        for box in boxes:
            cls = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            if cls != 0:  # person class only
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(overlay, f"{conf:.2f}", (x1, max(y1 - 5, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

        # --------- 3) Counts and overcrowding (red text) ---------
        color = (0, 0, 255)

        cv2.putText(overlay, f"C1: {cell_counts[0]:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C2: {cell_counts[1]:.1f}", (mid_x + 20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C3: {cell_counts[2]:.1f}", (20, mid_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(overlay, f"C4: {cell_counts[3]:.1f}", (mid_x + 20, mid_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)

        total_count = sum(cell_counts)
        cv2.putText(overlay, f"Total: {total_count:.1f}", (20, frame_height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)

        overcrowded_cells = [i+1 for i, c in enumerate(cell_counts) if c > CELL_THRESHOLD]
        if overcrowded_cells:
            warning_text = f"WARNING: Overcrowding in cells: {overcrowded_cells}"
            cv2.putText(overlay, warning_text, (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)

        # Write to video file
        out.write(overlay)

        # ------- Show realtime window (even if slow) -------
        cv2.imshow("Crowd Monitoring (MC-CNN + YOLO)", overlay)
        # waitKey controls display speed; increase if too fast, keep 1 if you don't care about speed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Processing complete, saved output video at {output_video_path}")

if __name__ == "__main__":
    main()


Using device: cpu
MC-CNN model loaded.
YOLO model loaded for person boxes.
